In [3]:
import pandas as pd
import json
import importlib

# load the data

In [4]:
concept_root = "../data/concept/"
out_concept_root = "../data/outside_concept/"
response_root = "../data/respondent/"

In [ ]:
# take the concepts 
with open(concept_root + 'new_cid_concept_us_food.json', 'r') as f:
    food_concepts = json.load(f)

# all concepts 
all_us_food_concepts = pd.read_excel(out_concept_root + '0407_cleaned_us_food_concepts.xlsx')

# open transformed
with open(response_root + 'transformed_0407_id_normal_interview.json', 'r', encoding='utf-8') as f:
    transformed_respondent = json.load(f)

response_table = pd.read_excel(response_root + "response_table_us_food.xlsx")
response_table.drop(columns=['id'], inplace=True)

In [ ]:
import sys
sys.path.append("../")
from models import similar as sm
from models import need_filter as nf

c:\Users\Yuding.Duan\OneDrive - Ipsos\3. self_projects\llm_synthetic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [159]:
import importlib
importlib.reload(nf)
importlib.reload(sm)

<module 'models.similar' from 'c:\\Users\\Yuding.Duan\\OneDrive - Ipsos\\3. self_projects\\llm_synthetic\\combination\\..\\models\\similar.py'>

# go for the whole process

**prepare all the data**

- food_concepts  
- all_us_food_concepts  
- transformed_respondent

In [ ]:
us_food_cates = list(set(all_us_food_concepts.dropna(subset=['CAT2'])['CAT2']))
# Example usage with caching:
corpus = us_food_cates[:]
# Initialize searcher and fit with cache (first run computes, subsequent runs load from cache)
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl"


In [ ]:
all_us_food_concepts.drop_duplicates(subset=['ConceptText'], inplace=True)
all_us_food_concepts.reset_index(drop=True, inplace=True)

In [11]:
kpi_mapping = {
    "relevance": "RelFlag",
    "differentiation": "DiffFlag",
    "believability": "BelFlag"
}

let's go

**ob_type**: `real` or `synthetic_contra` or `synthetic_same`

In [ ]:
# parameters 
CACHE_PATH = "../data/embeddings_cache/us_food_concepts.pkl" # vector cache path
cate_match_bound = 0.5 # this is for category matching
searcher_cate = sm.SimilaritySearcher()
searcher_cate.fit(us_food_cates, cache_path=CACHE_PATH)


## boundary stuff 
contra_lower_bound = 0.28
same_lower_bound = 0.5
smae_upper_bound = 0.75

Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 22 embeddings loaded from cache


In [121]:
final_result = {'ids':[], 'concept':[], 'question':[], 'answer':[], 'ob_type':[], 'corresponding_concept':[]}
final_results = [] 

In [ ]:
# minimum unit 
id = "b01f6cb0-0754-11f0-a000-d10a7e1714e4"
kpi = "differentiation"





C:\Users\Yuding.Duan\AppData\Local\Temp\ipykernel_32184\3734395815.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['ob_type'] = 'real'
C:\Users\Yuding.Duan\AppData\Local\Temp\ipykernel_32184\3734395815.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub['corresponding_concept'] = sub['concept']


In [ ]:
mask = (response_table['question'] == kpi) &(response_table['ids']==id)
sub = response_table[mask]
sub['ob_type'] = 'real'
sub['corresponding_concept'] = sub['concept']
final_results.append(sub)

In [152]:
if len(sub['answer'].value_counts()) == 1:
    process_type = 'contra' # to get the opposite answer
else:
    process_type = 'same' # to get the same answer

In [ ]:

for i in range(len(sub)):
    item = sub.iloc[i]

    # to get the category 
    if process_type == 'contra':
        query = food_concepts[str(item['concept'])]['concept_Cate']
        top_results, bottom_results = searcher_cate.search(query, top_n=5, bottom_m=0)
        suitable_cates = set([k[0] for k in top_results if k[1] >= cate_match_bound])

        if item['answer'] == 'yes':
            out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['L', 'ML']) & (all_us_food_concepts['CAT2'].isin(suitable_cates))
        else:
            out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['H', 'MH']) & (all_us_food_concepts['CAT2'].isin(suitable_cates))
        
        filted_concepts = all_us_food_concepts[out_mask]
        corpus = list(filted_concepts['ConceptText'])


        searcher_concept = sm.SimilaritySearcher()
        searcher_concept.fit(corpus, cache_path=CACHE_PATH)

        query = food_concepts[str(item['concept'])]['concept_content']
        top_results, bottom_results = searcher_concept.search(query, top_n=5, bottom_m=5)
        candidate = [item[0] for item  in bottom_results if item[1]<= contra_lower_bound]
        if not candidate:
            continue

        items = []
        respondent_info = transformed_respondent[id]
        reasoning = False

        for concept in candidate:
            items.append({
                "new_concept": concept,
                "kpi_type": kpi,
                "system_info": respondent_info,
                "return_reasoning": reasoning
            })

        # Run batch processing concurrently (use await in notebooks)
        print(f"Processing {len(items)} items concurrently...")
        batch_results = await nf.ai_filter_batch_async(
            items,
            max_concurrency=4,
            show_progress=False,
            progress_desc="AI filter"
        )
        
        if item['answer'] == "yes":
            pick_answer = 'no'    
        else:
            pick_answer = 'yes'
        
        for k in range(len(batch_results)):
            if batch_results[k] == pick_answer:
                final_result['ids'].append(id)
                final_result['concept'].append(candidate[k])
                final_result['question'].append(kpi)
                final_result['answer'].append(batch_results[k])
                final_result['ob_type'].append('synthetic_contra')
                final_result['corresponding_concept'].append(str(item['concept']))
    else: 

        if item['answer'] == 'yes':
            out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['H', 'MH'])
        else:
            out_mask = all_us_food_concepts[kpi_mapping[kpi]].isin(['L', 'ML']) 
        
        filted_concepts = all_us_food_concepts[out_mask]
        corpus = list(filted_concepts['ConceptText'])


        searcher_concept = sm.SimilaritySearcher()
        searcher_concept.fit(corpus, cache_path=CACHE_PATH)

        query = food_concepts[str(item['concept'])]['concept_content']
        top_results, bottom_results = searcher_concept.search(query, top_n=8, bottom_m=5)
        candidate = [item[0] for item  in top_results if same_lower_bound<= item[1]<= smae_upper_bound]
        if not candidate:
            continue

        items = []
        respondent_info = transformed_respondent[id]
        reasoning = False

        for concept in candidate:
            items.append({
                "new_concept": concept,
                "kpi_type": kpi,
                "system_info": respondent_info,
                "return_reasoning": reasoning
            })

        # Run batch processing concurrently (use await in notebooks)
        print(f"Processing {len(items)} items concurrently...")
        batch_results = await nf.ai_filter_batch_async(
            items,
            max_concurrency=4,
            show_progress=False,
            progress_desc="AI filter"
        )
        
        pick_answer = item['answer']
        for k in range(len(batch_results)):
            if batch_results[k] == pick_answer:
                final_result['ids'].append(id)
                final_result['concept'].append(candidate[k])
                final_result['question'].append(kpi)
                final_result['answer'].append(batch_results[k])
                final_result['ob_type'].append('synthetic_same')
                final_result['corresponding_concept'].append(str(item['concept']))


Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 1993 embeddings loaded from cache
Loading embeddings cache: ../data/embeddings_cache/us_food_concepts.pkl
Loaded 3962 cached embeddings
All 531 embeddings loaded from cache
Processing 5 items concurrently...


AI filter: 100%|██████████| 5/5 [00:26<00:00,  5.27s/it]


In [156]:
fr_df = pd.DataFrame(final_result)